# Phase 6 — Prediction Pipeline

This notebook builds a reusable 3-in-1 prediction pipeline for the AI Mental Health Assessment Platform.

Outputs:
- Mental Health Condition
- Severity Level
- Recommended Treatment


## Step 1 — Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np


## Step 2 — Define Project Paths


In [ ]:
ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = ROOT_DIR / "models"
REPORTS_DIR = ROOT_DIR / "reports"
SRC_DIR = ROOT_DIR / "src"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT_DIR:", ROOT_DIR)
print("DATA_DIR:", DATA_DIR)
print("MODELS_DIR:", MODELS_DIR)


## Step 3 — Define Target Configuration


In [ ]:
TARGET_CONFIG = {
    "condition": {
        "model_file": "condition_best_model.pkl",
        "preprocessor_file": "condition_preprocessor.pkl",
        "label": "Mental Health Condition"
    },
    "severity": {
        "model_file": "severity_best_model.pkl",
        "preprocessor_file": "severity_preprocessor.pkl",
        "label": "Severity Level"
    },
    "treatment": {
        "model_file": "treatment_best_model.pkl",
        "preprocessor_file": "treatment_preprocessor.pkl",
        "label": "Recommended Treatment"
    }
}


## Step 4 — Define Input Feature Columns

These columns must match the features used during Phase 2 preprocessing and Phase 3 training.


In [ ]:
FEATURE_COLUMNS = [
    "age",
    "gender",
    "occupation",
    "sleep_hours",
    "sleep_quality",
    "social_media_hours",
    "academic_work_pressure",
    "physical_activity_days",
    "stress_level",
    "anxiety_score",
    "depression_score",
    "work_life_balance",
    "mood_score",
    "concentration_level",
    "social_support",
]

NUMERIC_FEATURES = [
    "age",
    "sleep_hours",
    "sleep_quality",
    "social_media_hours",
    "academic_work_pressure",
    "physical_activity_days",
    "stress_level",
    "anxiety_score",
    "depression_score",
    "work_life_balance",
    "mood_score",
    "concentration_level",
    "social_support",
]

CATEGORICAL_FEATURES = [
    "gender",
    "occupation",
]


## Step 5 — Load Models and Preprocessors


In [ ]:
def load_artifact(path):
    if not path.exists():
        raise FileNotFoundError(f"Required artifact not found: {path}")
    return joblib.load(path)

artifacts = {}

for target_key, config in TARGET_CONFIG.items():
    model_path = MODELS_DIR / config["model_file"]
    preprocessor_path = MODELS_DIR / config["preprocessor_file"]

    artifacts[target_key] = {
        "model": load_artifact(model_path),
        "preprocessor": load_artifact(preprocessor_path),
        "label": config["label"],
    }

    print(f"Loaded {target_key}:")
    print("  Model:", model_path.name)
    print("  Preprocessor:", preprocessor_path.name)


## Step 6 — Create Input Validation Function


In [ ]:
def validate_input(input_data):
    if isinstance(input_data, dict):
        input_df = pd.DataFrame([input_data])
    elif isinstance(input_data, pd.DataFrame):
        input_df = input_data.copy()
    else:
        raise TypeError("input_data must be a dictionary or pandas DataFrame.")

    missing_columns = [col for col in FEATURE_COLUMNS if col not in input_df.columns]
    if missing_columns:
        raise ValueError(f"Missing input columns: {missing_columns}")

    input_df = input_df[FEATURE_COLUMNS].copy()

    for col in NUMERIC_FEATURES:
        input_df[col] = pd.to_numeric(input_df[col], errors="coerce")

    if input_df[NUMERIC_FEATURES].isnull().any().any():
        bad_cols = input_df[NUMERIC_FEATURES].columns[input_df[NUMERIC_FEATURES].isnull().any()].tolist()
        raise ValueError(f"Numeric columns contain invalid values: {bad_cols}")

    for col in CATEGORICAL_FEATURES:
        input_df[col] = input_df[col].astype(str)

    return input_df


## Step 7 — Create Single Target Prediction Function


In [ ]:
def predict_single_target(input_df, target_key):
    if target_key not in artifacts:
        raise ValueError(f"Unknown target_key: {target_key}")

    model = artifacts[target_key]["model"]
    preprocessor = artifacts[target_key]["preprocessor"]

    X_processed = preprocessor.transform(input_df)
    prediction = model.predict(X_processed)

    result = {
        "prediction": prediction[0]
    }

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_processed)[0]
        classes = model.classes_

        probability_dict = {
            str(cls): round(float(prob), 4)
            for cls, prob in zip(classes, probabilities)
        }

        result["confidence"] = round(float(np.max(probabilities)), 4)
        result["probabilities"] = probability_dict
    else:
        result["confidence"] = None
        result["probabilities"] = None

    return result


## Step 8 — Create 3-in-1 Prediction Function


In [ ]:
def predict_mental_health(input_data):
    input_df = validate_input(input_data)

    condition_result = predict_single_target(input_df, "condition")
    severity_result = predict_single_target(input_df, "severity")
    treatment_result = predict_single_target(input_df, "treatment")

    final_result = {
        "mental_health_condition": condition_result["prediction"],
        "condition_confidence": condition_result["confidence"],

        "severity": severity_result["prediction"],
        "severity_confidence": severity_result["confidence"],

        "recommended_treatment": treatment_result["prediction"],
        "treatment_confidence": treatment_result["confidence"],

        "details": {
            "condition_probabilities": condition_result["probabilities"],
            "severity_probabilities": severity_result["probabilities"],
            "treatment_probabilities": treatment_result["probabilities"],
        }
    }

    return final_result


## Step 9 — Test with a Sample User Input


In [ ]:
sample_user = {
    "age": 24,
    "gender": "Male",
    "occupation": "Student",
    "sleep_hours": 5.5,
    "sleep_quality": 4,
    "social_media_hours": 6,
    "academic_work_pressure": 8,
    "physical_activity_days": 1,
    "stress_level": 8,
    "anxiety_score": 7,
    "depression_score": 6,
    "work_life_balance": 3,
    "mood_score": 4,
    "concentration_level": 4,
    "social_support": 3,
}

prediction = predict_mental_health(sample_user)
prediction


## Step 10 — Pretty Print Prediction Output


In [ ]:
def print_prediction_report(result):
    print("AI Mental Health Assessment Result")
    print("=" * 45)
    print(f"Mental Health Condition : {result['mental_health_condition']}")
    print(f"Condition Confidence    : {result['condition_confidence']}")
    print("-" * 45)
    print(f"Severity Level          : {result['severity']}")
    print(f"Severity Confidence     : {result['severity_confidence']}")
    print("-" * 45)
    print(f"Recommended Treatment   : {result['recommended_treatment']}")
    print(f"Treatment Confidence    : {result['treatment_confidence']}")

print_prediction_report(prediction)


## Step 11 — Batch Prediction Function


In [ ]:
def predict_batch(input_df):
    validated_df = validate_input(input_df)

    results = []

    for _, row in validated_df.iterrows():
        row_result = predict_mental_health(row.to_dict())
        results.append(row_result)

    output_df = pd.DataFrame([
        {
            "mental_health_condition": item["mental_health_condition"],
            "condition_confidence": item["condition_confidence"],
            "severity": item["severity"],
            "severity_confidence": item["severity_confidence"],
            "recommended_treatment": item["recommended_treatment"],
            "treatment_confidence": item["treatment_confidence"],
        }
        for item in results
    ])

    return output_df


## Step 12 — Test Batch Prediction


In [ ]:
sample_batch = pd.DataFrame([
    sample_user,
    {
        "age": 31,
        "gender": "Female",
        "occupation": "Professional",
        "sleep_hours": 7.5,
        "sleep_quality": 8,
        "social_media_hours": 2,
        "academic_work_pressure": 4,
        "physical_activity_days": 4,
        "stress_level": 3,
        "anxiety_score": 2,
        "depression_score": 2,
        "work_life_balance": 8,
        "mood_score": 8,
        "concentration_level": 8,
        "social_support": 7,
    }
])

batch_predictions = predict_batch(sample_batch)
batch_predictions


## Step 13 — Save Prediction Output Example


In [ ]:
prediction_output_path = REPORTS_DIR / "sample_prediction_output.json"

with open(prediction_output_path, "w", encoding="utf-8") as f:
    json.dump(prediction, f, indent=4, ensure_ascii=False)

print("Saved:", prediction_output_path)


## Step 14 — Generate `src/predict.py`

This file will be used later by FastAPI and Streamlit.


In [ ]:
predict_py_code = r'''
from pathlib import Path
import joblib
import pandas as pd
import numpy as np


ROOT_DIR = Path(__file__).resolve().parents[1]
MODELS_DIR = ROOT_DIR / "models"

TARGET_CONFIG = {
    "condition": {
        "model_file": "condition_best_model.pkl",
        "preprocessor_file": "condition_preprocessor.pkl",
        "label": "Mental Health Condition"
    },
    "severity": {
        "model_file": "severity_best_model.pkl",
        "preprocessor_file": "severity_preprocessor.pkl",
        "label": "Severity Level"
    },
    "treatment": {
        "model_file": "treatment_best_model.pkl",
        "preprocessor_file": "treatment_preprocessor.pkl",
        "label": "Recommended Treatment"
    }
}

FEATURE_COLUMNS = [
    "age",
    "gender",
    "occupation",
    "sleep_hours",
    "sleep_quality",
    "social_media_hours",
    "academic_work_pressure",
    "physical_activity_days",
    "stress_level",
    "anxiety_score",
    "depression_score",
    "work_life_balance",
    "mood_score",
    "concentration_level",
    "social_support",
]

NUMERIC_FEATURES = [
    "age",
    "sleep_hours",
    "sleep_quality",
    "social_media_hours",
    "academic_work_pressure",
    "physical_activity_days",
    "stress_level",
    "anxiety_score",
    "depression_score",
    "work_life_balance",
    "mood_score",
    "concentration_level",
    "social_support",
]

CATEGORICAL_FEATURES = [
    "gender",
    "occupation",
]


def load_artifact(path):
    if not path.exists():
        raise FileNotFoundError(f"Required artifact not found: {path}")
    return joblib.load(path)


def load_prediction_artifacts():
    artifacts = {}

    for target_key, config in TARGET_CONFIG.items():
        model_path = MODELS_DIR / config["model_file"]
        preprocessor_path = MODELS_DIR / config["preprocessor_file"]

        artifacts[target_key] = {
            "model": load_artifact(model_path),
            "preprocessor": load_artifact(preprocessor_path),
            "label": config["label"],
        }

    return artifacts


ARTIFACTS = None


def get_artifacts():
    global ARTIFACTS
    if ARTIFACTS is None:
        ARTIFACTS = load_prediction_artifacts()
    return ARTIFACTS


def validate_input(input_data):
    if isinstance(input_data, dict):
        input_df = pd.DataFrame([input_data])
    elif isinstance(input_data, pd.DataFrame):
        input_df = input_data.copy()
    else:
        raise TypeError("input_data must be a dictionary or pandas DataFrame.")

    missing_columns = [col for col in FEATURE_COLUMNS if col not in input_df.columns]
    if missing_columns:
        raise ValueError(f"Missing input columns: {missing_columns}")

    input_df = input_df[FEATURE_COLUMNS].copy()

    for col in NUMERIC_FEATURES:
        input_df[col] = pd.to_numeric(input_df[col], errors="coerce")

    if input_df[NUMERIC_FEATURES].isnull().any().any():
        bad_cols = input_df[NUMERIC_FEATURES].columns[input_df[NUMERIC_FEATURES].isnull().any()].tolist()
        raise ValueError(f"Numeric columns contain invalid values: {bad_cols}")

    for col in CATEGORICAL_FEATURES:
        input_df[col] = input_df[col].astype(str)

    return input_df


def predict_single_target(input_df, target_key):
    artifacts = get_artifacts()

    if target_key not in artifacts:
        raise ValueError(f"Unknown target_key: {target_key}")

    model = artifacts[target_key]["model"]
    preprocessor = artifacts[target_key]["preprocessor"]

    X_processed = preprocessor.transform(input_df)
    prediction = model.predict(X_processed)

    result = {
        "prediction": prediction[0]
    }

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_processed)[0]
        classes = model.classes_

        result["confidence"] = round(float(np.max(probabilities)), 4)
        result["probabilities"] = {
            str(cls): round(float(prob), 4)
            for cls, prob in zip(classes, probabilities)
        }
    else:
        result["confidence"] = None
        result["probabilities"] = None

    return result


def predict_mental_health(input_data):
    input_df = validate_input(input_data)

    condition_result = predict_single_target(input_df, "condition")
    severity_result = predict_single_target(input_df, "severity")
    treatment_result = predict_single_target(input_df, "treatment")

    return {
        "mental_health_condition": condition_result["prediction"],
        "condition_confidence": condition_result["confidence"],
        "severity": severity_result["prediction"],
        "severity_confidence": severity_result["confidence"],
        "recommended_treatment": treatment_result["prediction"],
        "treatment_confidence": treatment_result["confidence"],
        "details": {
            "condition_probabilities": condition_result["probabilities"],
            "severity_probabilities": severity_result["probabilities"],
            "treatment_probabilities": treatment_result["probabilities"],
        }
    }


def predict_batch(input_df):
    validated_df = validate_input(input_df)
    results = []

    for _, row in validated_df.iterrows():
        results.append(predict_mental_health(row.to_dict()))

    return pd.DataFrame([
        {
            "mental_health_condition": item["mental_health_condition"],
            "condition_confidence": item["condition_confidence"],
            "severity": item["severity"],
            "severity_confidence": item["severity_confidence"],
            "recommended_treatment": item["recommended_treatment"],
            "treatment_confidence": item["treatment_confidence"],
        }
        for item in results
    ])
'''

predict_py_path = SRC_DIR / "predict.py"

with open(predict_py_path, "w", encoding="utf-8") as f:
    f.write(predict_py_code)

print("Saved:", predict_py_path)


## Step 15 — Test `src/predict.py` Import


In [ ]:
import sys

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.predict import predict_mental_health as production_predict

production_result = production_predict(sample_user)
production_result


## Step 16 — Save Prediction Pipeline Report


In [ ]:
report_path = REPORTS_DIR / "prediction_pipeline_report.md"

report_text = f'''# Phase 6 — Prediction Pipeline Report

## Objective

Build a reusable prediction pipeline for the AI Mental Health Assessment Platform.

## Inputs

The pipeline expects the following input features:

{FEATURE_COLUMNS}

## Outputs

The pipeline returns three predictions:

1. Mental Health Condition
2. Severity Level
3. Recommended Treatment

## Saved Production File

- src/predict.py

## Sample Output

```json
{json.dumps(prediction, indent=4, ensure_ascii=False)}
```

## Next Phase

Phase 7 will build a FastAPI backend using `src/predict.py`.
'''

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("Saved:", report_path)


## Step 17 — Final Conclusion

Phase 6 is complete. The project now has a reusable prediction pipeline that can be used by FastAPI, Streamlit, and batch prediction scripts.
